# 📊 Module 5.4: Model Evaluation with `mlflow.evaluate()`

**Goal**: Use MLFlow's built-in evaluation framework for systematic model assessment.

---

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

mlflow.autolog(disable=True)
mlflow.set_experiment("05_model_evaluation")

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Setup complete!")

## 1. Basic Evaluation with `mlflow.evaluate()`

MLFlow's evaluate function computes metrics and generates visualizations automatically.

In [ ]:
with mlflow.start_run(run_name="evaluate_demo"):
    # Train model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Log model
    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(model, "model", signature=signature)
    
    run_id = mlflow.active_run().info.run_id
    
    # Create evaluation dataset
    eval_data = X_test.copy()
    eval_data["target"] = y_test
    
    # Evaluate!
    results = mlflow.evaluate(
        model=f"runs:/{run_id}/model",
        data=eval_data,
        targets="target",
        model_type="classifier",  # Tells MLFlow which metrics to compute
    )
    
    print("📊 Evaluation Results:")
    print("=" * 50)
    for metric, value in sorted(results.metrics.items()):
        if isinstance(value, float):
            print(f"  {metric}: {value:.4f}")
        else:
            print(f"  {metric}: {value}")
    
    print("\n✅ Evaluation complete! Check MLFlow UI for artifacts.")
    print("   The UI will show confusion matrix, per-class metrics, and more.")

## 2. View Evaluation Artifacts

`mlflow.evaluate()` automatically logs these artifacts:
- Confusion matrix plot
- Per-class metrics table
- Summary metrics

In [ ]:
# View the evaluation artifacts
if results.artifacts:
    print("📁 Evaluation Artifacts:")
    for name, artifact in results.artifacts.items():
        print(f"  - {name}: {type(artifact).__name__}")
else:
    print("Artifacts are saved to the MLFlow run — check the UI!")

## 3. Custom Metrics

In [ ]:
from mlflow.metrics import make_metric
from sklearn.metrics import balanced_accuracy_score

# Define a custom metric function
def balanced_acc(eval_df, _builtin_metrics):
    """Calculate balanced accuracy."""
    return balanced_accuracy_score(
        eval_df["target"], 
        eval_df["prediction"]
    )

# Create the custom metric
balanced_accuracy_metric = make_metric(
    eval_fn=balanced_acc,
    greater_is_better=True,
    name="balanced_accuracy"
)

with mlflow.start_run(run_name="evaluate_custom_metrics"):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(model, "model", signature=signature)
    
    run_id = mlflow.active_run().info.run_id
    
    eval_data = X_test.copy()
    eval_data["target"] = y_test
    
    # Evaluate with custom metrics
    results = mlflow.evaluate(
        model=f"runs:/{run_id}/model",
        data=eval_data,
        targets="target",
        model_type="classifier",
        extra_metrics=[balanced_accuracy_metric],  # ← Custom metrics!
    )
    
    print("📊 Results with custom metrics:")
    for metric, value in sorted(results.metrics.items()):
        marker = "⭐" if "balanced" in metric else "  "
        if isinstance(value, float):
            print(f"{marker} {metric}: {value:.4f}")
        else:
            print(f"{marker} {metric}: {value}")

## 🔑 Key Takeaways

1. **`mlflow.evaluate()`** automates metric computation and artifact generation
2. Set **`model_type="classifier"`** or **`"regressor"`** for built-in metrics
3. **Custom metrics** with `make_metric()` extend the built-in evaluator
4. Evaluation artifacts (plots, tables) are **automatically logged** to the run
5. Great for **comparing models** — run evaluate on multiple models, then compare in the UI

---

## ➡️ Next: `05_exercises.ipynb`